## HumanInTheLoopMiddleware中间件
在 工具调用前 中断Agent运行，等待用户对工具调用请求决策。可选的决策有：`approve（同意执行）`、 `edit（编辑调用配置后执行）`、 `reject（拒绝执行）`

### 参数说明
#### interrupt_on —工具名和中断策略的映射
策略可以是True、False或InterruptOnConfig对象，精细控制决策选项。
- True表示所有决策(approve, edit, reject) 都可以选择，
- False表示不中断，即无需审批即可执行。
  
InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
① allowed_decisions 精细控制中断后允许的决策。
② description ：特定工具的中断描述信息，优先级高于description_prefix，后 者会更
改 所有 工具中断的描述。

#### 参数2：description_prefix —自定义中断描述
默认为 "Tool execution requires approval" ，下面的举例可以看到效果

In [6]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦",
                },
            },
            description_prefix="中断啦",
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "请帮我查询今天北京的天气"
                    "查询今日新闻"
                    "查看ID为 'sk2131421' 的邮件内容，"
                    "向15641685664@qq.com发送邮件，标题是'哈哈哈'，"
                    "内容是：'你好啊'"
                    "同时做这四件事"
                )
            )
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)

print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)

# print("==== 逐个打印 interrupt 请求 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='55201698-fb50-4874-b5bc-8d5020b09deb'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Task 1: 
Query today\'s weather in Beijing (北京)\n   - Task 2: Query today\'s news (今日新闻)\n   - Task 3: Read email with
ID \'sk2131421\'\n   - Task 4: Send an email to \'15641685664@qq.com\' with subject \'哈哈哈\' and body 
\'你好啊\'\n   - The user explicitly says "同时做这四件事" (do these four things at the same time).\n\n2.  
**Identify Required Tools & Parameters:**\n   - Tool 1: `get_weather`\n     - `city`: "北京"\n     - `is_forcast`: 
Not specified, defaults to false. I\'ll leave it as default or explicitly set to false.\n   - Tool 2: `get_news`\n 
- No parameters required.\n   - Tool 3: `read_email_tool`\n     - `email_id`: "sk2131421"\n   - Tool 4: 
`send_email_tool`\n     - `recipient`: "15641685664@qq.com"\n     - `subject`: "哈哈哈"\n     - `body`: 
"你好啊"\n\n3.  **Construct Tool Calls:**\n   All four tool calls can be made concurrently since they are 
independent.\n   ```json\n   [\n     {"name": "get_weather", "arguments": {"city": "北京"}},\n     {"name": 
"get_news", "arguments": {}},\n     {"name": "read_email_tool", "arguments": {"email_id": "sk2131421"}},\n     
{"name": "send_email_tool", "arguments": {"recipient": "15641685664@qq.com", "subject": "哈哈哈", "body": 
"你好啊"}}\n   ]\n   ```\n\n4.  **Execute & Format Output:**\n   I will generate the tool calls as specified. The 
system will handle the execution and return results. I just need to output the function calling format.\n\n   Wait,
let\'s double-check the parameters for `get_weather`. The schema says `is_forcast` is optional with default false. 
I\'ll omit it or set it to false. It\'s fine either way.\n   All other parameters match exactly.\n   I will proceed
with the function call.✅\n   Output matches the required format.✅\n   Ready. \n   Self-Correction/Verification 
during thought:\n   - `get_weather(city="北京")` -> correct\n   - `get_news()` -> correct\n   - 
`read_email_tool(email_id="sk2131421")` -> correct\n   - `send_email_tool(recipient="15641685664@qq.com", 
subject="哈哈哈", body="你好啊")` -> correct\n   All good. I will generate the response now. \n   Note: The prompt 
asks to do them simultaneously, which aligns with calling multiple tools in one turn. I will output the tool calls 
directly.✅\n   Proceed. \n   [Output Generation] -> Function calls block. \n   *(Self-Correction/Refinement during
generation prep)*: I should make sure the JSON structure matches the expected format for the AI assistant. I\'ll 
output the standard tool call format.\n   All parameters are correctly typed and required ones are provided.\n   
Done. \n   Output matches the response. \n   [Proceeds]'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 909,
                    'prompt_tokens': 542,
                    'total_tokens': 1451,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 766,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 909
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 542}
                },
                'model_provider': 'dashscope',
                'model_name': 'qwen3.6-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-9a147759-3edd-96fc-bd51-ebcf34077bb7',
                'finish_reason': '

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_4e861b815cb44a52adb569ae)
 Call ID: call_4e861b815cb44a52adb569ae
  Args:
    city: 北京
  get_news (call_47c7e613410a43d2b0105caf)
 Call ID: call_47c7e613410a43d2b0105caf
  Args:
  read_email_tool (call_f4a33c7518a449978e031dd3)
 Call ID: call_f4a33c7518a449978e031dd3
  Args:
    email_id: sk2131421
  send_email_tool (call_d88c52e449114e568b30cdb4)
 Call ID: call_d88c52e449114e568b30cdb4
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京'}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='85b3d4318214328d8ca0e377bc26428c'
    )
]

{
    'name': 'get_weather',
    'args': {'city': '北京'},
    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京'}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦'
}

In [8]:
# 指明工具调用请求决策
# 如果有中断，说明进入人在环了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国北京市", "is_forcast": True}
}
}
news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}

# 决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)
        
if interrupts:
    # 审批通过
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config, # 必须是同一个 thread_id
    )
    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
==== 审批后继续执行 ====
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_4e861b815cb44a52adb569ae)
 Call ID: call_4e861b815cb44a52adb569ae
  Args:
    city: 中国北京市
    is_forcast: True
  get_news (call_47c7e613410a43d2b0105caf)
 Call ID: call_47c7e613410a43d2b0105caf
  Args:
  read_email_tool (call_f4a33c7518a449978e031dd3)
 Call ID: call_f4a33c7518a449978e031dd3
  Args:
    email_id: sk2131421
  send_email_tool (call_d88c52e449114e568b30cdb4)
 Call ID: call_d88c52e449114e568b30cdb4
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

中国北京市今天天气不错
明天下雨
================================= Tool Message ============================